# SprintBoard · GRPO Fine-tuning on Hugging Face TRL

This is the **judge-runnable training notebook** for the SprintBoard environment
(OpenEnv Hackathon submission).

**What this notebook does**, end-to-end, against the *real* environment:

1. Clones the SprintBoard Space and installs the env in-process.
2. Evaluates a **baseline** policy on all 15 sprint-planning tasks.
3. Fine-tunes `Qwen/Qwen2.5-1.5B-Instruct` with **GRPO + LoRA** using a
   *full multi-step trajectory* reward.
4. Re-evaluates the **trained** policy on the same 15 tasks.
5. Saves before/after reward plots to `assets/` and prints the deltas.

**Hardware**: tested on a free Colab T4 (≈15 GB VRAM). One full pass takes
30–60 minutes. Reduce `NUM_EPOCHS` / `MAX_PROMPTS` if you just want a smoke
test.

## 1 · Setup — clone the SprintBoard env and install training deps

Re-running the cell is safe; the clone step only fires the first time.

In [ ]:
import os, sys, subprocess, pathlib

REPO_DIR = pathlib.Path("sprint_planning_env_repo")
if not REPO_DIR.exists() and not pathlib.Path("server").exists():
    !git clone -q https://huggingface.co/spaces/vikramsrini/sprint_planning_env {REPO_DIR}
if REPO_DIR.exists():
    os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

!pip install -q -U pip
!pip install -q -r requirements-train.txt
!pip install -q -e .
print("✅ Setup complete")

## 2 · Imports & global config

In [ ]:
import json, random, re, time
from pathlib import Path
from statistics import mean
from typing import List, Dict

import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

from sprint_planning_env.models import SprintAction
from sprint_planning_env.server.environment import SprintBoardEnvironment
from sprint_planning_env.server.tasks import TASK_REGISTRY, list_task_ids

# ----- Knobs -----------------------------------------------------------------
MODEL_ID         = "Qwen/Qwen2.5-1.5B-Instruct"  # 1.5B fits on a free T4 with LoRA
TASK_IDS         = list_task_ids()                # all 15 tasks
EPISODES_PER_TASK = 1                             # eval rollouts per task
NUM_GENERATIONS  = 4                              # GRPO group size
MAX_NEW_TOKENS   = 384                            # enough for ~10-12 commands
MAX_PROMPT_TOK   = 768
NUM_EPOCHS       = 3                              # bump to 6-8 for stronger gains
LR               = 3e-6
OUT_DIR          = Path("./sprintboard_qwen_lora")
ASSETS_DIR       = Path("./assets"); ASSETS_DIR.mkdir(exist_ok=True)

print(f"Tasks: {len(TASK_IDS)} | model: {MODEL_ID} | torch: {torch.__version__}")

## 3 · Multi-step rollout + reward function

The agent outputs a **plan**: one command per line. We execute the whole plan
in the SprintBoard environment and use the deterministic grader's final score
as the reward signal. This is genuine multi-step credit assignment — the
policy is rewarded for *reasoning across the full episode*, not for emitting
a syntactically valid first token.

Two shaping signals are combined:

* **Format reward** — 0.10 if the first non-empty line is a valid SprintBoard
  verb. Keeps the model from collapsing onto markdown / explanations.
* **Trajectory reward** — final grader score after running every command in
  the rollout (clamped to [0, 1]).

In [ ]:
VALID_VERBS = {
    "LIST_BACKLOG", "VIEW_STORY", "CHECK_DEPS", "VIEW_TEAM", "VIEW_VELOCITY",
    "VIEW_SPRINT", "VIEW_BUGS", "VIEW_EPIC", "SEARCH_BACKLOG", "ESTIMATE",
    "ASSIGN", "UNASSIGN", "ADD_TO_SPRINT", "REMOVE_FROM_SPRINT", "SET_PRIORITY",
    "FLAG_RISK", "DECOMPOSE", "FINALIZE_SPRINT",
}

_TASK_RE = re.compile(r"TASK_ID:\s*(task_\d+)")


def extract_task_id(prompt: str) -> str:
    m = _TASK_RE.search(prompt)
    return m.group(1) if m else "task_1"


def parse_plan(text: str, max_cmds: int = 15) -> List[str]:
    """Pull command lines out of a free-form completion."""
    if isinstance(text, list):           # GRPO sometimes hands us [{role, content}]
        text = text[0] if text and isinstance(text[0], str) else \
               (text[0].get("content", "") if text and isinstance(text[0], dict) else "")
    cmds: List[str] = []
    for raw in str(text).splitlines():
        line = raw.strip().strip("`").strip("-*>• ").strip()
        if not line:
            continue
        verb = line.split()[0].upper()
        if verb in VALID_VERBS:
            cmds.append(line)
        if len(cmds) >= max_cmds:
            break
    return cmds


def rollout_score(task_id: str, plan: List[str]) -> Dict[str, float]:
    """Run a plan against the env and return reward + metadata."""
    env = SprintBoardEnvironment()
    env.reset(task_id=task_id, seed=0)
    cumulative = 0.0
    last_grader = 0.0
    finalized = False
    n_steps = 0
    for cmd in plan or ["FINALIZE_SPRINT"]:
        obs = env.step(SprintAction(command=cmd))
        cumulative += float(obs.reward or 0.0)
        g = obs.metadata.get("grader_score")
        if g is not None:
            last_grader = float(g)
        n_steps += 1
        if obs.done:
            finalized = True
            break
    if not finalized:                                # auto-finalize if model never did
        obs = env.step(SprintAction(command="FINALIZE_SPRINT"))
        g = obs.metadata.get("grader_score")
        if g is not None:
            last_grader = float(g)
        n_steps += 1
    return {
        "reward": float(np.clip(last_grader, 0.0, 1.0)),
        "steps": n_steps,
        "cumulative": cumulative,
    }


# ----- TRL reward callbacks (called once per generation) ----------------------
training_reward_log: List[float] = []

def trajectory_reward(prompts, completions, **kwargs):
    rewards = []
    for prompt, completion in zip(prompts, completions):
        task_id = extract_task_id(str(prompt))
        plan = parse_plan(completion)
        rewards.append(rollout_score(task_id, plan)["reward"])
    if rewards:
        training_reward_log.append(float(mean(rewards)))
    return rewards


def format_reward(prompts, completions, **kwargs):
    rewards = []
    for completion in completions:
        plan = parse_plan(completion, max_cmds=1)
        rewards.append(0.10 if plan else 0.0)
    return rewards


# Smoke test
demo = rollout_score("task_1", ["LIST_BACKLOG", "ESTIMATE 103 5", "ESTIMATE 106 3",
                                 "ESTIMATE 107 3", "FINALIZE_SPRINT"])
print("Smoke-test rollout on task_1 →", demo)

## 4 · Build training dataset — one prompt per task

Each prompt embeds the task scenario the model would see in episode step 0.
Repeating the 15 prompts across epochs gives GRPO multiple shots at each
scenario.

In [ ]:
SYSTEM_PROMPT = """You are an expert Agile Scrum Master operating SprintBoard.
Given a sprint-planning scenario, produce an ordered PLAN of commands that
investigates the board and fixes the issues, then finalises the sprint.
Output **one command per line**, no markdown, no commentary, no numbering.
Always end the plan with FINALIZE_SPRINT.

Allowed commands:
  Investigation: LIST_BACKLOG, VIEW_STORY <id>, CHECK_DEPS <id>, VIEW_TEAM,
                 VIEW_VELOCITY, VIEW_SPRINT, VIEW_BUGS, VIEW_EPIC <id>,
                 SEARCH_BACKLOG <kw>
  Planning:      ESTIMATE <id> <pts>, ASSIGN <id> <Name>, UNASSIGN <id>,
                 ADD_TO_SPRINT <id>, REMOVE_FROM_SPRINT <id>,
                 SET_PRIORITY <id> <P0|P1|P2>, FLAG_RISK <id> <reason>,
                 DECOMPOSE <epic_id> \"sub1\" \"sub2\" ...,
                 FINALIZE_SPRINT

Strategy:
  1. Investigate first (LIST_BACKLOG, VIEW_TEAM, VIEW_VELOCITY ...).
  2. Then fix the planning issue revealed by the scenario alert.
  3. End with FINALIZE_SPRINT."""


def build_prompt(task_id: str) -> str:
    task = TASK_REGISTRY[task_id]
    return (
        f"{SYSTEM_PROMPT}\n\n"
        f"TASK_ID: {task_id}  (difficulty: {task['difficulty']})\n"
        f"SCENARIO ALERT: {task['alert']}\n\n"
        f"Produce the full plan now (one command per line):"
    )


prompts = [build_prompt(tid) for tid in TASK_IDS]
dataset = Dataset.from_dict({"prompt": prompts})
print(f"Built dataset with {len(dataset)} prompts (1 per task).")

## 5 · Load model + LoRA adapter

In [ ]:
print(f"Loading {MODEL_ID} …")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
)
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 6 · Baseline evaluation (pre-training)

Runs 1 rollout per task with greedy decoding and records the grader score.

In [ ]:
@torch.inference_mode()
def policy_rollout(task_id: str, max_new_tokens: int = MAX_NEW_TOKENS,
                   temperature: float = 0.2) -> Dict[str, float]:
    prompt = build_prompt(task_id)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=MAX_PROMPT_TOK).to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=max(temperature, 1e-3),
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
    )
    completion = tokenizer.decode(output[0][inputs.input_ids.shape[1]:],
                                  skip_special_tokens=True)
    plan = parse_plan(completion)
    rec = rollout_score(task_id, plan)
    rec.update({"plan": plan, "completion": completion[:600]})
    return rec


def evaluate(label: str) -> Dict[str, Dict]:
    results = {}
    print(f"\n=== {label} evaluation ({len(TASK_IDS)} tasks) ===")
    for tid in TASK_IDS:
        rec = policy_rollout(tid)
        results[tid] = rec
        print(f"  {tid:<8s}  reward={rec['reward']:.3f}  steps={rec['steps']:>2d}  "
              f"plan_len={len(rec['plan'])}")
    avg = mean(r["reward"] for r in results.values())
    print(f"--- {label} mean grader score: {avg:.3f}")
    return results


baseline_results = evaluate("BASELINE (pre-training)")

## 7 · GRPO training

In [ ]:
training_reward_log.clear()

config = GRPOConfig(
    output_dir=str(OUT_DIR),
    learning_rate=LR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_NEW_TOKENS,
    max_prompt_length=MAX_PROMPT_TOK,
    num_train_epochs=NUM_EPOCHS,
    logging_steps=1,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to="none",
    save_strategy="no",
    dataloader_num_workers=0,
    remove_unused_columns=False,
    beta=0.04,
    seed=42,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[trajectory_reward, format_reward],
    args=config,
    train_dataset=dataset,
)

t0 = time.time()
train_result = trainer.train()
print(f"\nTraining finished in {time.time()-t0:.1f}s")

## 8 · Trained-policy evaluation

In [ ]:
trained_results = evaluate("TRAINED (post-GRPO)")

## 9 · Plots & summary saved to `assets/`

Two PNGs are saved so the README can embed them:

* `assets/grpo_reward_curve.png` — mean reward per GRPO update.
* `assets/before_after_per_task.png` — baseline vs trained per task.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(range(1, len(training_reward_log) + 1), training_reward_log,
        marker="o", linewidth=2, color="#7c3aed", label="GRPO group reward")
if training_reward_log:
    rolling = np.convolve(training_reward_log,
                          np.ones(min(5, len(training_reward_log)))/min(5, len(training_reward_log)),
                          mode="valid")
    ax.plot(range(len(training_reward_log) - len(rolling) + 1,
                  len(training_reward_log) + 1),
            rolling, color="#22c55e", linewidth=2, label="5-step rolling mean")
ax.set_xlabel("GRPO update step")
ax.set_ylabel("Mean trajectory reward (grader score)")
ax.set_ylim(0, 1.0)
ax.set_title("SprintBoard · GRPO training reward curve")
ax.grid(alpha=0.3, linestyle="--")
ax.legend()
fig.tight_layout()
fig.savefig(ASSETS_DIR / "grpo_reward_curve.png", dpi=180)
plt.show()

task_ids = list(baseline_results.keys())
base_vals = [baseline_results[t]["reward"] for t in task_ids]
train_vals = [trained_results[t]["reward"] for t in task_ids]
delta = mean(train_vals) - mean(base_vals)

fig, ax = plt.subplots(figsize=(11, 4.6))
x = np.arange(len(task_ids))
w = 0.38
ax.bar(x - w/2, base_vals, w, label=f"Baseline ({mean(base_vals):.2f})",
       color="#94a3b8", edgecolor="#475569")
ax.bar(x + w/2, train_vals, w, label=f"Trained ({mean(train_vals):.2f})",
       color="#7c3aed", edgecolor="#5b21b6")
ax.set_xticks(x); ax.set_xticklabels([t.replace("task_","T") for t in task_ids])
ax.set_ylim(0, 1.0)
ax.set_xlabel("Task"); ax.set_ylabel("Grader score (0–1)")
ax.set_title(f"SprintBoard · baseline vs GRPO-trained Qwen2.5-1.5B  (Δ = {delta:+.2f})")
ax.grid(axis="y", alpha=0.3, linestyle="--"); ax.legend()
fig.tight_layout()
fig.savefig(ASSETS_DIR / "before_after_per_task.png", dpi=180)
plt.show()

summary = {
    "model": MODEL_ID,
    "epochs": NUM_EPOCHS,
    "baseline_mean": round(mean(base_vals), 4),
    "trained_mean":  round(mean(train_vals), 4),
    "delta":         round(delta, 4),
    "per_task": {t: {"baseline": round(b, 4), "trained": round(tr, 4)}
                  for t, b, tr in zip(task_ids, base_vals, train_vals)},
}
(ASSETS_DIR / "training_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print("\nPlots saved →", ASSETS_DIR.resolve())

## 10 · (Optional) Push the LoRA adapter to the Hub

In [ ]:
PUSH = False  # flip to True after `huggingface-cli login`
REPO_ID = "your-username/sprintboard-qwen25-1.5b-grpo"

if PUSH:
    model.push_to_hub(REPO_ID, private=False)
    tokenizer.push_to_hub(REPO_ID)
    print("✅ Pushed to", REPO_ID)
else:
    print("Set PUSH = True and authenticate to publish the adapter.")